<a href="https://colab.research.google.com/github/HansMacias/Actividad-pacientes/blob/main/MLP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
import pandas as pd

# URL del archivo CSV en GitHub
csv_url = 'https://raw.githubusercontent.com/Fernoguez/EEG/main/DM.csv'

# Cargar el archivo CSV en un DataFrame de pandas
df = pd.read_csv(csv_url)

# Mostrar las primeras 5 filas del DataFrame para inspeccionar los datos
display(df.head())

,Grupo,Participante,Minuto,C3,C4,CZ,EMG,F3,F4,F7,...,O1,O2,P3,P4,PZ,ROG,T3,T4,T5,T6
0,0,EMV,1,1.457191,1.465362,1.389850,1.469966,1.471585,1.383370,1.416681,...,1.419505,1.460171,1.435626,1.401177,1.405440,1.190718,1.486129,1.455536,1.457664,1.464154
1,0,EMV,10,1.442564,1.372664,1.361794,1.310307,1.477391,1.366440,1.439025,...,1.417780,1.421081,1.410483,1.395753,1.393475,1.116700,1.474367,1.413287,1.477452,1.412524
2,0,EMV,11,1.445738,1.397893,1.363018,1.246467,1.451693,1.372738,1.398308,...,1.444138,1.434578,1.425999,1.411785,1.403913,1.118753,1.472099,1.433305,1.484939,1.440299
3,0,EMV,12,1.381500,1.422462,1.443394,1.445102,1.422085,1.413301,1.450779,...,1.427840,1.451817,1.403597,1.434047,1.428170,1.167026,1.428117,1.453502,1.373695,1.437340
4,0,EMV,13,1.443665,1.355041,1.360214,1.301061,1.474152,1.365256,1.433039,...,1.415445,1.402281,1.391482,1.395755,1.400138,1.139237,1.464603,1.404249,1.453814,1.405438


In [8]:
# Explorar la columna 'Grupo'
print("Valores únicos en 'Grupo':", df['Grupo'].unique())
print("Distribución de 'Grupo':")
display(df['Grupo'].value_counts())

# Explorar otras columnas potencialmente categóricas
print("\nValores únicos en 'Participante':", df['Participante'].unique())
print("Distribución de 'Participante':")
display(df['Participante'].value_counts())

Valores únicos en 'Grupo': [0 1]
Distribución de 'Grupo':


,count
Grupo,
0,500
1,400



Valores únicos en 'Participante': ['EMV' 'GH2' 'GUR' 'JAL' 'JAN' 'MGN' 'MJN' 'MMA' 'RAN' 'VCN' 'AEF' 'CLM'
 'FGV' 'JGM' 'LIV' 'PCM' 'RLM' 'RRM']
Distribución de 'Participante':


,count
Participante,
EMV,50
GH2,50
GUR,50
JAL,50
JAN,50
MGN,50
MJN,50
MMA,50
RAN,50


In [11]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import classification_report, accuracy_score
from sklearn.impute import SimpleImputer # Importar SimpleImputer
import numpy as np

# Identificar características (X) y variable objetivo (y)
# Asumimos que 'Grupo' es la variable objetivo.
# Excluimos 'Participante' y 'Minuto' por ahora, y 'ROG' ya que en algunos contextos puede ser un canal de referencia o un artefacto.
# El resto de las columnas numéricas son los canales EEG.

X = df.drop(columns=['Grupo', 'Participante', 'Minuto'])
y = df['Grupo']

# Dividir los datos en conjuntos de entrenamiento y prueba
# Usaremos un 80% para entrenamiento y un 20% para prueba.
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# *** Paso de imputación de valores faltantes ***
# Inicializar el imputador con la estrategia de la media
imputer = SimpleImputer(strategy='mean')

# Ajustar el imputador SOLO en el conjunto de entrenamiento y transformar ambos conjuntos
X_train_imputed = imputer.fit_transform(X_train)
X_test_imputed = imputer.transform(X_test)

# Escalar las características (ahora sobre los datos imputados)
# StandardScaler es bueno para MLPs ya que centra los datos en 0 y los escala a una varianza unitaria.
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_imputed)
X_test_scaled = scaler.transform(X_test_imputed)

print("Datos preparados, imputados y escalados.")
print(f"Tamaño del conjunto de entrenamiento: {X_train_scaled.shape}")
print(f"Tamaño del conjunto de prueba: {X_test_scaled.shape}")

Datos preparados, imputados y escalados.
Tamaño del conjunto de entrenamiento: (720, 22)
Tamaño del conjunto de prueba: (180, 22)


In [15]:
# Re-ejecutar el entrenamiento del modelo MLP
# hidden_layer_sizes: define la estructura de las capas ocultas (e.g., (100, 50) significa 2 capas con 100 y 50 neuronas respectivamente)
# max_iter: número máximo de épocas (iteraciones) para el entrenamiento
# random_state: para reproducibilidad
mlp = MLPClassifier(hidden_layer_sizes=(100, 50), max_iter=300, activation='relu', solver='adam', random_state=42)

# Entrenar el modelo con los datos escalados e imputados
mlp.fit(X_train_scaled, y_train)

print("Modelo MLP entrenado exitosamente.")

# Re-ejecutar la evaluación del modelo
y_pred = mlp.predict(X_test_scaled)

# Evaluar el rendimiento del modelo
accuracy = accuracy_score(y_test, y_pred)
report = classification_report(y_test, y_pred)

print(f"Precisión (Accuracy) del modelo MLP: {accuracy:.4f}")
print("\nInforme de Clasificación:")
print(report)

from google.colab import files

# Crear un DataFrame con las etiquetas reales y las predicciones
results_df = pd.DataFrame({'True_Label': y_test, 'Predicted_Label': y_pred})

# Guardar el DataFrame en un archivo CSV
file_name = 'eeg_mlp_predictions.csv'
results_df.to_csv(file_name, index=False)

# Ofrecer el archivo para descargar
files.download(file_name)

print(f"Archivo '{file_name}' generado y listo para descargar. Contiene las etiquetas reales y las predicciones del modelo MLP.")

Modelo MLP entrenado exitosamente.
Precisión (Accuracy) del modelo MLP: 0.9167

Informe de Clasificación:
              precision    recall  f1-score   support

           0       0.90      0.95      0.93       100
           1       0.93      0.88      0.90        80

    accuracy                           0.92       180
   macro avg       0.92      0.91      0.92       180
weighted avg       0.92      0.92      0.92       180



<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Archivo 'eeg_mlp_predictions.csv' generado y listo para descargar. Contiene las etiquetas reales y las predicciones del modelo MLP.


In [14]:
# Re-ejecutar el entrenamiento del modelo MLP
# hidden_layer_sizes: define la estructura de las capas ocultas (e.g., (100, 50) significa 2 capas con 100 y 50 neuronas respectivamente)
# max_iter: número máximo de épocas (iteraciones) para el entrenamiento
# random_state: para reproducibilidad
mlp = MLPClassifier(hidden_layer_sizes=(100, 50), max_iter=300, activation='relu', solver='adam', random_state=42)

# Entrenar el modelo con los datos escalados e imputados
mlp.fit(X_train_scaled, y_train)

print("Modelo MLP entrenado exitosamente.")

# Re-ejecutar la evaluación del modelo
y_pred = mlp.predict(X_test_scaled)

# Evaluar el rendimiento del modelo
accuracy = accuracy_score(y_test, y_pred)
report = classification_report(y_test, y_pred)

print(f"Precisión (Accuracy) del modelo MLP: {accuracy:.4f}")
print("\nInforme de Clasificación:")
print(report)

from google.colab import files

# Crear un DataFrame con las etiquetas reales y las predicciones
results_df = pd.DataFrame({'True_Label': y_test, 'Predicted_Label': y_pred})

# Guardar el DataFrame en un archivo CSV
file_name = 'eeg_mlp_predictions.csv'
results_df.to_csv(file_name, index=False)

# Ofrecer el archivo para descargar
files.download(file_name)

print(f"Archivo '{file_name}' generado y listo para descargar. Contiene las etiquetas reales y las predicciones del modelo MLP.")

Modelo MLP entrenado exitosamente.
Precisión (Accuracy) del modelo MLP: 0.9167

Informe de Clasificación:
              precision    recall  f1-score   support

           0       0.90      0.95      0.93       100
           1       0.93      0.88      0.90        80

    accuracy                           0.92       180
   macro avg       0.92      0.91      0.92       180
weighted avg       0.92      0.92      0.92       180



<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Archivo 'eeg_mlp_predictions.csv' generado y listo para descargar. Contiene las etiquetas reales y las predicciones del modelo MLP.


In [13]:
# Re-ejecutar el entrenamiento del modelo MLP
# hidden_layer_sizes: define la estructura de las capas ocultas (e.g., (100, 50) significa 2 capas con 100 y 50 neuronas respectivamente)
# max_iter: número máximo de épocas (iteraciones) para el entrenamiento
# random_state: para reproducibilidad
mlp = MLPClassifier(hidden_layer_sizes=(100, 50), max_iter=300, activation='relu', solver='adam', random_state=42)

# Entrenar el modelo con los datos escalados e imputados
mlp.fit(X_train_scaled, y_train)

print("Modelo MLP entrenado exitosamente.")

# Re-ejecutar la evaluación del modelo
y_pred = mlp.predict(X_test_scaled)

# Evaluar el rendimiento del modelo
accuracy = accuracy_score(y_test, y_pred)
report = classification_report(y_test, y_pred)

print(f"Precisión (Accuracy) del modelo MLP: {accuracy:.4f}")
print("\nInforme de Clasificación:")
print(report)

from google.colab import files

# Crear un DataFrame con las etiquetas reales y las predicciones
results_df = pd.DataFrame({'True_Label': y_test, 'Predicted_Label': y_pred})

# Guardar el DataFrame en un archivo CSV
file_name = 'eeg_mlp_predictions.csv'
results_df.to_csv(file_name, index=False)

# Ofrecer el archivo para descargar
files.download(file_name)

print(f"Archivo '{file_name}' generado y listo para descargar. Contiene las etiquetas reales y las predicciones del modelo MLP.")

Modelo MLP entrenado exitosamente.
Precisión (Accuracy) del modelo MLP: 0.9167

Informe de Clasificación:
              precision    recall  f1-score   support

           0       0.90      0.95      0.93       100
           1       0.93      0.88      0.90        80

    accuracy                           0.92       180
   macro avg       0.92      0.91      0.92       180
weighted avg       0.92      0.92      0.92       180



<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Archivo 'eeg_mlp_predictions.csv' generado y listo para descargar. Contiene las etiquetas reales y las predicciones del modelo MLP.


In [12]:
# Re-ejecutar el entrenamiento del modelo MLP
# hidden_layer_sizes: define la estructura de las capas ocultas (e.g., (100, 50) significa 2 capas con 100 y 50 neuronas respectivamente)
# max_iter: número máximo de épocas (iteraciones) para el entrenamiento
# random_state: para reproducibilidad
mlp = MLPClassifier(hidden_layer_sizes=(100, 50), max_iter=300, activation='relu', solver='adam', random_state=42)

# Entrenar el modelo con los datos escalados e imputados
mlp.fit(X_train_scaled, y_train)

print("Modelo MLP entrenado exitosamente.")

# Re-ejecutar la evaluación del modelo
y_pred = mlp.predict(X_test_scaled)

# Evaluar el rendimiento del modelo
accuracy = accuracy_score(y_test, y_pred)
report = classification_report(y_test, y_pred)

print(f"Precisión (Accuracy) del modelo MLP: {accuracy:.4f}")
print("\nInforme de Clasificación:")
print(report)

from google.colab import files

# Crear un DataFrame con las etiquetas reales y las predicciones
results_df = pd.DataFrame({'True_Label': y_test, 'Predicted_Label': y_pred})

# Guardar el DataFrame en un archivo CSV
file_name = 'eeg_mlp_predictions.csv'
results_df.to_csv(file_name, index=False)

# Ofrecer el archivo para descargar
files.download(file_name)

print(f"Archivo '{file_name}' generado y listo para descargar. Contiene las etiquetas reales y las predicciones del modelo MLP.")

Modelo MLP entrenado exitosamente.
Precisión (Accuracy) del modelo MLP: 0.9167

Informe de Clasificación:
              precision    recall  f1-score   support

           0       0.90      0.95      0.93       100
           1       0.93      0.88      0.90        80

    accuracy                           0.92       180
   macro avg       0.92      0.91      0.92       180
weighted avg       0.92      0.92      0.92       180



<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Archivo 'eeg_mlp_predictions.csv' generado y listo para descargar. Contiene las etiquetas reales y las predicciones del modelo MLP.


Ahora que los datos están listos, vamos a entrenar el modelo MLP. Configuraré un MLP básico con algunas capas ocultas y una función de activación común.

In [10]:
# Inicializar y entrenar el modelo MLP
# hidden_layer_sizes: define la estructura de las capas ocultas (e.g., (100, 50) significa 2 capas con 100 y 50 neuronas respectivamente)
# max_iter: número máximo de épocas (iteraciones) para el entrenamiento
# random_state: para reproducibilidad
mlp = MLPClassifier(hidden_layer_sizes=(100, 50), max_iter=300, activation='relu', solver='adam', random_state=42)

# Entrenar el modelo con los datos escalados
mlp.fit(X_train_scaled, y_train)

print("Modelo MLP entrenado exitosamente.")

ValueError: Input X contains NaN.
MLPClassifier does not accept missing values encoded as NaN natively. For supervised learning, you might want to consider sklearn.ensemble.HistGradientBoostingClassifier and Regressor which accept missing values encoded as NaNs natively. Alternatively, it is possible to preprocess the data, for instance by using an imputer transformer in a pipeline or drop samples with missing values. See https://scikit-learn.org/stable/modules/impute.html You can find a list of all estimators that handle NaN values at the following page: https://scikit-learn.org/stable/modules/impute.html#estimators-that-handle-nan-values

El modelo ya está entrenado. A continuación, evaluaremos su rendimiento en el conjunto de prueba para ver qué tan bien generaliza a datos no vistos y generaremos el informe de clasificación.

In [ ]:
# Realizar predicciones en el conjunto de prueba escalado
y_pred = mlp.predict(X_test_scaled)

# Evaluar el rendimiento del modelo
accuracy = accuracy_score(y_test, y_pred)
report = classification_report(y_test, y_pred)

print(f"Precisión (Accuracy) del modelo MLP: {accuracy:.4f}")
print("\nInforme de Clasificación:")
print(report)

Finalmente, para tu análisis, generaré un archivo CSV descargable que contiene las etiquetas reales y las predicciones del modelo en el conjunto de prueba.

In [ ]:
from google.colab import files

# Crear un DataFrame con las etiquetas reales y las predicciones
results_df = pd.DataFrame({'True_Label': y_test, 'Predicted_Label': y_pred})

# Guardar el DataFrame en un archivo CSV
file_name = 'eeg_mlp_predictions.csv'
results_df.to_csv(file_name, index=False)

# Ofrecer el archivo para descargar
files.download(file_name)

print(f"Archivo '{file_name}' generado y listo para descargar. Contiene las etiquetas reales y las predicciones del modelo MLP.")

Con esta información, podremos definir mejor el problema de clasificación y qué preprocesamiento será necesario antes de entrenar un modelo.

Ahora que tenemos los datos cargados, podemos comenzar con el preprocesamiento y la preparación para la clasificación. Una vez que vea la estructura de los datos, podremos discutir las posibles estrategias de clasificación.